# 24 m spatially buffered split generator

Geometry-only code. Generates a candidate split from the dataset. Compare against the saved training memberships before treating it as the historical split. Camera-centre distance is not footprint non-overlap.

Source: `graindiscoloration (2).ipynb`, zero-based cell 67.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
# ============================================================
# 24 m SPATIALLY BUFFERED LEAKAGE-AWARE SPLIT
# Expected: ~207 train / 40 val / 41 test / 112 excluded
# ============================================================

from pathlib import Path
import re
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")

# -----------------------------
# Configuration
# -----------------------------
BUFFER_M = 24.0
TARGET_VAL = 40
TARGET_TEST = 40

OUTPUT_DIR = Path("/kaggle/working/spatial_24m_split")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAT_LON_PATTERN = re.compile(
    r"lat_(-?\d+(?:\.\d+)?)_lon_(-?\d+(?:\.\d+)?)",
    re.IGNORECASE
)

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}


# ============================================================
# 1. Locate the RGB folder automatically
# ============================================================

preferred_rgb_dirs = [
    Path("/kaggle/input/datasets/hemal7/gd-rgbms-db/RGB"),
    Path("/kaggle/input/hemal7/gd-rgbms-db/RGB"),
    Path("/kaggle/input/jarin19/rgbmsdb/RGB"),
    Path("/kaggle/input/gd-rgbms-db/RGB"),
]

candidate_dirs = [p for p in preferred_rgb_dirs if p.exists()]

if not candidate_dirs:
    candidate_dirs = [
        p for p in Path("/kaggle/input").rglob("*")
        if p.is_dir() and p.name.lower() == "rgb"
    ]

if not candidate_dirs:
    raise FileNotFoundError(
        "Could not find the RGB folder under /kaggle/input."
    )


def count_valid_images(folder):
    count = 0
    for p in folder.rglob("*"):
        if (
            p.is_file()
            and p.suffix.lower() in VALID_EXTENSIONS
            and LAT_LON_PATTERN.search(p.stem)
        ):
            count += 1
    return count


candidate_info = [
    (folder, count_valid_images(folder))
    for folder in candidate_dirs
]

candidate_info = [item for item in candidate_info if item[1] > 0]

if not candidate_info:
    raise FileNotFoundError(
        "RGB folders were found, but no filenames containing "
        "latitude and longitude were detected."
    )

RGB_DIR, detected_count = max(candidate_info, key=lambda x: x[1])

print("Selected RGB directory:")
print(RGB_DIR)
print("Detected coordinate-labelled RGB images:", detected_count)


# ============================================================
# 2. Read filenames and extract GPS coordinates
# ============================================================

records = []

for image_path in sorted(RGB_DIR.rglob("*")):
    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in VALID_EXTENSIONS:
        continue

    match = LAT_LON_PATTERN.search(image_path.stem)

    if match is None:
        continue

    records.append({
        "sample_name": image_path.stem,
        "latitude": float(match.group(1)),
        "longitude": float(match.group(2)),
        "rgb_path": str(image_path),
    })

df = pd.DataFrame(records)

# Avoid accidental duplicate files
df = (
    df.drop_duplicates(subset="sample_name")
      .sort_values("sample_name")
      .reset_index(drop=True)
)

print("\nUnique samples:", len(df))

if len(df) != 400:
    print(
        f"WARNING: Expected 400 samples, but found {len(df)}. "
        "The code will continue using the detected samples."
    )


# ============================================================
# 3. Convert latitude/longitude to local metre coordinates
# ============================================================

EARTH_RADIUS_M = 6_371_000.0

mean_latitude_rad = np.deg2rad(df["latitude"].mean())
mean_longitude = df["longitude"].mean()
mean_latitude = df["latitude"].mean()

df["x_m"] = (
    np.deg2rad(df["longitude"] - mean_longitude)
    * EARTH_RADIUS_M
    * np.cos(mean_latitude_rad)
)

df["y_m"] = (
    np.deg2rad(df["latitude"] - mean_latitude)
    * EARTH_RADIUS_M
)

coordinates = df[["x_m", "y_m"]].to_numpy()


# ============================================================
# 4. Identify independent spatial fields/components
# ============================================================

distance_matrix = cdist(coordinates, coordinates)

adjacency = (
    (distance_matrix <= BUFFER_M)
    & (distance_matrix > 0)
)

number_of_components, component_labels = connected_components(
    csr_matrix(adjacency.astype(np.uint8)),
    directed=False,
    return_labels=True
)

df["spatial_component_24m"] = component_labels

component_sizes = (
    df.groupby("spatial_component_24m")
      .size()
      .sort_values()
)

print("\n24 m spatial components:")
print(component_sizes.to_string())

if number_of_components != 2:
    raise ValueError(
        f"Expected 2 major spatial components, but found "
        f"{number_of_components}. Please send the printed output "
        "before proceeding."
    )


# ============================================================
# 5. Find the best buffered end-region within each field
# ============================================================

def find_best_buffered_end_split(
    component_df,
    target_holdout,
    buffer_m=24.0,
    target_tolerance=3
):
    """
    Selects one end of a spatial field as validation/test.
    The opposite side becomes training.
    Images inside the buffer between them are excluded.
    """

    component_points = component_df[["x_m", "y_m"]].to_numpy()
    n = len(component_df)

    best_result = None

    # Search many spatial directions
    angles = np.linspace(0.0, 179.75, 1440)

    minimum_k = max(15, target_holdout - target_tolerance)
    maximum_k = min(n - 15, target_holdout + target_tolerance)

    for angle_degree in angles:
        angle_radian = np.deg2rad(angle_degree)

        direction = np.array([
            np.cos(angle_radian),
            np.sin(angle_radian)
        ])

        projected_position = component_points @ direction
        sorted_indices = np.argsort(projected_position)

        for side in ["low", "high"]:
            for k in range(minimum_k, maximum_k + 1):

                if side == "low":
                    holdout_local = sorted_indices[:k]
                    holdout_boundary = projected_position[
                        holdout_local
                    ].max()

                    train_local = np.where(
                        projected_position
                        > holdout_boundary + buffer_m
                    )[0]

                    excluded_local = np.where(
                        (projected_position > holdout_boundary)
                        & (
                            projected_position
                            <= holdout_boundary + buffer_m
                        )
                    )[0]

                else:
                    holdout_local = sorted_indices[-k:]
                    holdout_boundary = projected_position[
                        holdout_local
                    ].min()

                    train_local = np.where(
                        projected_position
                        < holdout_boundary - buffer_m
                    )[0]

                    excluded_local = np.where(
                        (
                            projected_position
                            >= holdout_boundary - buffer_m
                        )
                        & (
                            projected_position
                            < holdout_boundary
                        )
                    )[0]

                if len(train_local) == 0:
                    continue

                minimum_distance = cdist(
                    component_points[holdout_local],
                    component_points[train_local]
                ).min()

                if minimum_distance < buffer_m:
                    continue

                # Priority:
                # 1. Retain the maximum number of images
                # 2. Stay close to requested holdout size
                # 3. Prefer slightly larger separation
                score = (
                    len(holdout_local) + len(train_local),
                    -abs(len(holdout_local) - target_holdout),
                    minimum_distance
                )

                if (
                    best_result is None
                    or score > best_result["score"]
                ):
                    best_result = {
                        "score": score,
                        "angle_degree": angle_degree,
                        "side": side,
                        "holdout_local": holdout_local,
                        "train_local": train_local,
                        "excluded_local": excluded_local,
                        "minimum_distance": minimum_distance,
                    }

    if best_result is None:
        raise RuntimeError(
            "No valid buffered split was found for this component."
        )

    component_global_indices = component_df.index.to_numpy()

    best_result["holdout_global"] = component_global_indices[
        best_result["holdout_local"]
    ]

    best_result["train_global"] = component_global_indices[
        best_result["train_local"]
    ]

    best_result["excluded_global"] = component_global_indices[
        best_result["excluded_local"]
    ]

    return best_result


# Smaller field supplies validation; larger field supplies test.
# Both fields still contribute training images.
sorted_components = component_sizes.index.tolist()

validation_component = sorted_components[0]
test_component = sorted_components[1]

validation_result = find_best_buffered_end_split(
    df[df["spatial_component_24m"] == validation_component],
    target_holdout=TARGET_VAL,
    buffer_m=BUFFER_M
)

test_result = find_best_buffered_end_split(
    df[df["spatial_component_24m"] == test_component],
    target_holdout=TARGET_TEST,
    buffer_m=BUFFER_M
)


# ============================================================
# 6. Assign final split labels
# ============================================================

df["split"] = "excluded_buffer"

df.loc[validation_result["train_global"], "split"] = "train"
df.loc[test_result["train_global"], "split"] = "train"

df.loc[
    validation_result["holdout_global"],
    "split"
] = "validation"

df.loc[
    test_result["holdout_global"],
    "split"
] = "test"


# ============================================================
# 7. Verify that no cross-set pair is within 24 m
# ============================================================

def cross_split_statistics(split_a, split_b):
    indices_a = df.index[df["split"] == split_a].to_numpy()
    indices_b = df.index[df["split"] == split_b].to_numpy()

    distances = distance_matrix[np.ix_(indices_a, indices_b)]

    return {
        "split_a": split_a,
        "split_b": split_b,
        "minimum_distance_m": float(distances.min()),
        "pairs_at_or_below_24m": int(
            np.sum(distances <= BUFFER_M)
        ),
    }


verification = pd.DataFrame([
    cross_split_statistics("train", "validation"),
    cross_split_statistics("train", "test"),
    cross_split_statistics("validation", "test"),
])

print("\nCross-split verification:")
print(verification.to_string(index=False))

if not (verification["pairs_at_or_below_24m"] == 0).all():
    raise AssertionError(
        "Verification failed: at least one cross-set pair "
        "is within 24 m."
    )

print("\n✅ VERIFIED: No cross-set pair is within 24 m.")


# ============================================================
# 8. Save assignment and split files
# ============================================================

split_order = [
    "train",
    "validation",
    "test",
    "excluded_buffer"
]

summary = (
    df["split"]
    .value_counts()
    .reindex(split_order, fill_value=0)
    .rename_axis("split")
    .reset_index(name="number_of_images")
)

retained = int(
    summary.loc[
        summary["split"] != "excluded_buffer",
        "number_of_images"
    ].sum()
)

print("\nFinal split summary:")
print(summary.to_string(index=False))
print("\nTotal retained:", retained)
print("Total excluded:", int((df["split"] == "excluded_buffer").sum()))

assignment_columns = [
    "sample_name",
    "latitude",
    "longitude",
    "x_m",
    "y_m",
    "spatial_component_24m",
    "split",
    "rgb_path",
]

df[assignment_columns].to_csv(
    OUTPUT_DIR / "spatial_24m_split_assignments.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "spatial_24m_split_summary.csv",
    index=False
)

verification.to_csv(
    OUTPUT_DIR / "spatial_24m_split_verification.csv",
    index=False
)

for split_name in split_order:
    split_df = df[df["split"] == split_name][assignment_columns]

    split_df.to_csv(
        OUTPUT_DIR / f"{split_name}_spatial_24m.csv",
        index=False
    )


# ============================================================
# 9. Create spatial split map
# ============================================================

plot_styles = {
    "train": {
        "color": "#1f77b4",
        "marker": "o",
        "label": "Train"
    },
    "validation": {
        "color": "#ff7f0e",
        "marker": "^",
        "label": "Validation"
    },
    "test": {
        "color": "#2ca02c",
        "marker": "s",
        "label": "Test"
    },
    "excluded_buffer": {
        "color": "#bdbdbd",
        "marker": "x",
        "label": "Excluded buffer"
    },
}

fig, ax = plt.subplots(figsize=(10, 6.5))

for split_name in split_order:
    subset = df[df["split"] == split_name]
    style = plot_styles[split_name]

    ax.scatter(
        subset["x_m"],
        subset["y_m"],
        s=36 if split_name != "excluded_buffer" else 28,
        c=style["color"],
        marker=style["marker"],
        label=f'{style["label"]} (n={len(subset)})',
        alpha=0.90
    )

ax.set_title(
    "24 m Spatially Buffered Leakage-Aware Split",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Local east–west position (m)")
ax.set_ylabel("Local north–south position (m)")
ax.grid(alpha=0.20)
ax.legend(frameon=True)
ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout()

figure_path = OUTPUT_DIR / "spatial_24m_split_map.png"

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 10. ZIP all outputs for download
# ============================================================

zip_path = Path(
    "/kaggle/working/spatial_24m_leakage_aware_split.zip"
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for file_path in OUTPUT_DIR.iterdir():
        if file_path.is_file():
            zip_file.write(
                file_path,
                arcname=file_path.name
            )

print("\n" + "=" * 68)
print("24 m LEAKAGE-AWARE SPLIT COMPLETED")
print("=" * 68)
print("Output folder:", OUTPUT_DIR)
print("Download ZIP:", zip_path)

try:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
except Exception:
    pass